# NB3 · Building and evaluating a model

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---

Building the model takes fifteen lines in this notebook. The rest is given over to
measuring how much it is worth.

There are fewer check cells than in NB2. The checks here target silent errors only.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
               'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'en'


In [ ]:
# Fixed cell. Rebuilds the cohort you assembled in NB2.
import pipeline as pl

state = pl.prepare(verbose=False)
cohort = state['cohort']
features = state['features']
cohort = cohort.rename(columns={'prolonged_stay': 'target'})
print(f'Cohort ready: {len(cohort)} stays, {len(features)} features.')


---

## Step 1 · Split the data

A patient may have more than one intensive care stay in this cohort. If the data is
split randomly at row level, one stay of a patient falls into training and another into
test. The model recognises that patient, test performance rises, and the rise is not
real.

This is among the most common silent errors made by generative AI tools. Asked for a
split, the tool divides rows at random by default; taking the patient identifier into
account has to be requested explicitly.


### Prompt 1

```
There is a pandas DataFrame named cohort. Each row is one intensive care stay. The
subject_id column holds the patient identifier and the target column holds the binary
outcome. A patient may have more than one stay.

Write a single Python cell that splits the data into training and test sets. Make the
split at patient level: no patient may appear in both sets. Use a test fraction of
0.30.

CONTRACT
Produce two DataFrames named train and test.
No subject_id may appear in both sets.
The target column must take both 0 and 1 in each set.
Print the row count and event rate of each set.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 1


In [ ]:
checks.check_split(train, test, target='target', patient_id='subject_id')


---

## Step 2 · Build the model

Scalers, imputers and encoders must be fitted on the training set alone and only
applied to the test set. Where those steps run across the whole dataset before the
split, values from the test set are carried into training.

This error raises no warning, improves performance, and looks reasonable when the code
is read. Placing the entire preprocessing chain inside a single Pipeline makes it
structurally impossible.


### Prompt 2

```
Build a baseline model using the train and test DataFrames. Write a single Python
cell.

Apply median imputation and scaling to numeric columns, most frequent imputation and
one-hot encoding to categorical columns. Use logistic regression as the classifier
with balanced class weights.

Do not use subject_id, hadm_id, stay_id, intime or target as features.

Do not search hyperparameters. A baseline exists to be beaten, not to be good.

CONTRACT
Produce a fitted scikit-learn Pipeline named model.
Every preprocessing step must sit inside that Pipeline; no fit call may occur
outside it.
Produce a list named feature_list holding the column names used.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 2


In [ ]:
checks.check_model(model)


---

## Step 3 · Produce predictions

The model must return probabilities rather than class labels. A threshold is set on the
probability and justified clinically. A model that returns class labels directly has
fixed the threshold at its own default, and that default is not a clinical decision.


### Prompt 3

```
Produce predictions on the test set with the fitted model. Write a single Python cell.

CONTRACT
Produce an array named probability holding the positive class probability for every
row in the test set.
Its length must equal the number of rows in the test set.
Its values must lie between 0 and 1.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 3


In [ ]:
checks.check_predictions(probability, n_expected=len(test))


---

## The accuracy trap

The two cells below are fixed. Run the first and read the figure, then run the second.


In [ ]:
accuracy = ((probability >= 0.5).astype(int) == test['target'].values).mean()
print(f'Test accuracy: {accuracy:.1%}')


In [ ]:
null = ev.null_comparison(test['target'])
print(f"A rule that learns nothing: {null['accuracy']:.1%}")
print()
print('The gap between the two figures is what the model actually contributes.')


In an imbalanced clinical problem, accuracy measures how rare the condition is rather
than what the model does. At a prevalence of seven percent a rule that always predicts
the negative class is ninety three percent accurate.


---

## The evaluation report

The cell below is the fixed evaluation section of this notebook and applies six
headings in order: Discrimination with a confidence interval, calibration, the
operating point, clinical translation, the subgroup breakdown and a null comparison.

The `target_sensitivity` value corresponds to the cost balance on your problem card. It
is held high where a miss is the costlier error. This is a clinical decision rather than
a technical default.


In [ ]:
report = ev.honest_report(
    test['target'], probability,
    groups=test['gender'] if 'gender' in test.columns else None,
    target_sensitivity=0.80,
    label='baseline model · prolonged intensive care stay',
)


In [ ]:
figure = ev.plot_curves(test['target'], probability)


## Reading the report

If the confidence interval includes 0.5, the model cannot be distinguished from chance,
whatever the point estimate says.

If the calibration slope falls below one, the model is over-confident. For a clinician
setting a threshold, this means the threshold does not sit where it appears to.

The clinical translation line gives the number of alerts fired per hundred patients and
how many of them are true. It is the counterpart here of the Epic Sepsis Model example
from the lecture.

In the subgroup breakdown some groups carry the words insufficient sample instead of a
figure. That is a finding rather than a shortcoming: a system cannot be shown to be fair
for a group it was never tested on.

The report is expected to come out unfavourable in this scenario. On a cohort of one
hundred patients the confidence interval is wide and the model's advantage over the null
comparison is small. Reaching that conclusion about a system you built yourself is a
different experience from hearing about someone else's failure.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
